# Display the architecture (No. of stages, channels per stage) for each model

In [ ]:
# ============================================================================
# Display the architecture (# of stages, channels per stage) for each model
# ============================================================================
import torch
import torch.nn as nn
import timm
import gc



def analyze_backbone_structure(backbone: nn.Module, 
                               img_size: int, 
                               model_name: str = "Model") -> dict:
    """
    Comprehensively analyze the feature extraction stages of a backbone.
    
    Creates a probe input, runs it through backbone, and reports:
      • Number of stages/blocks
      • Channel dimensions per stage
      • Spatial dimensions (H, W) per stage
      • Feature map count
    
    Returns dictionary with:
      {
        'num_stages': int,
        'stage_dims': list of channel counts,
        'stage_spatial': list of (H, W) tuples,
        'total_features': int (H × W × C for stage),
      }
    """
    print(f"\n{'─'*70}")
    print(f"  BACKBONE STRUCTURE ANALYSIS: {model_name}")
    print(f"{'─'*70}")
    print(f"  Input image size: {img_size}×{img_size}")
    
    with torch.no_grad():
        # Create PROBE input (explicit naming instead of "dummy")
        probe_input = torch.randn(1, 3, img_size, img_size)
        
        # Forward pass to extract features
        feature_maps = backbone(probe_input)
        
        num_stages = len(feature_maps)
        print(f"  Number of stages: {num_stages}")
        print()
        
        stage_dims = []
        stage_spatial = []
        
        # Detailed per-stage analysis
        print(f"  {'Stage':<8} {'Channels':<12} {'Spatial size':<20} {'Feature count':<15}")
        print(f"  {'─'*8}─{'─'*12}─{'─'*20}─{'─'*15}")
        
        for i, feat in enumerate(feature_maps):
            b, c, h, w = feat.shape
            stage_dims.append(c)
            stage_spatial.append((h, w))
            
            downsampling = img_size / h  # E.g., 224/56 = 4x downsampling
            total_features = c * h * w
            
            print(f"  Stage {i:<2} {c:<12} {h:>3}×{w:<3} ({downsampling:.0f}x down) {total_features:<15,}")
        
        del probe_input, feature_maps
    
    print(f"{'─'*70}\n")
    
    return {
        'num_stages': num_stages,
        'stage_dims': stage_dims,
        'stage_spatial': stage_spatial,
    }





print("\n" + "█"*80)
print("█" + "  BACKBONE STRUCTURES: ALL 5 MODELS".center(78) + "█")
print("█"*80 + "\n")

# Store info for each backbone
all_backbones = {}



# Model 1: EfficientNet-B5
print("Loading Model 1...")
backbone_eff = timm.create_model('efficientnet_b5', pretrained=True, 
                                  features_only=True, num_classes=0)
all_backbones['EfficientNet-B5'] = analyze_backbone_structure(
    backbone_eff, 456, "EfficientNet-B5")
del backbone_eff



# Model 2: Inception-V3
print("\nLoading Model 2...")
backbone_inc = timm.create_model('inception_v3', pretrained=True,
                                  features_only=True, num_classes=0)
all_backbones['Inception-V3'] = analyze_backbone_structure(
    backbone_inc, 299, "Inception-V3")
del backbone_inc



# Model 3: ConvNeXtV2-Tiny
print("\nLoading Model 3...")
backbone_cnx = timm.create_model('convnextv2_tiny', pretrained=True,
                                  features_only=True, num_classes=0)
all_backbones['ConvNeXtV2-Tiny'] = analyze_backbone_structure(
    backbone_cnx, 224, "ConvNeXtV2-Tiny")
del backbone_cnx



# Model 4: DenseNet-201
print("\nLoading Model 4...")
backbone_dns = timm.create_model('densenet201', pretrained=True,
                                  features_only=True, num_classes=0)
all_backbones['DenseNet-201'] = analyze_backbone_structure(
    backbone_dns, 256, "DenseNet-201")
del backbone_dns




# Model 5: ResNeXt-50 (32×4d)
print("\nLoading Model 5...")
backbone_rnx = timm.create_model('resnext50_32x4d', pretrained=True,
                                  features_only=True, num_classes=0)
all_backbones['ResNeXt-50 (32×4d)'] = analyze_backbone_structure(
    backbone_rnx, 256, "ResNeXt-50 (32×4d)")
del backbone_rnx





# ── Print Summary Table ───────────────────────────────────────────────────────
print("\n" + "="*100)
print("  QUICK REFERENCE: ALL BACKBONES")
print("="*100)
print()
print(f"{'Model Name':<25} {'# Stages':<12} {'Channel Dimensions':<65}")
print("-"*100)

for name, info in all_backbones.items():
    num_stages = info['num_stages']
    channels = info['stage_dims']
    channels_str = " → ".join(str(c) for c in channels)
    print(f"{name:<25} {num_stages:<12} {channels_str}")

print("="*100)
print()

# Clean up
gc.collect()
torch.cuda.empty_cache()

print("✅ Complete! All 5 backbones analyzed and ready for model initialization")


████████████████████████████████████████████████████████████████████████████████
█                       BACKBONE STRUCTURES: ALL 5 MODELS                      █
████████████████████████████████████████████████████████████████████████████████

Loading Model 1...


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



──────────────────────────────────────────────────────────────────────
  BACKBONE STRUCTURE ANALYSIS: EfficientNet-B5
──────────────────────────────────────────────────────────────────────
  Input image size: 456×456
  Number of stages: 5

  Stage    Channels     Spatial size         Feature count  
  ──────────────────────────────────────────────────────────
  Stage 0  24           228×228 (2x down) 1,247,616      
  Stage 1  40           114×114 (4x down) 519,840        
  Stage 2  64            57×57  (8x down) 207,936        
  Stage 3  176           29×29  (16x down) 148,016        
  Stage 4  512           15×15  (30x down) 115,200        
──────────────────────────────────────────────────────────────────────


Loading Model 2...


model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]


──────────────────────────────────────────────────────────────────────
  BACKBONE STRUCTURE ANALYSIS: Inception-V3
──────────────────────────────────────────────────────────────────────
  Input image size: 299×299
  Number of stages: 5

  Stage    Channels     Spatial size         Feature count  
  ──────────────────────────────────────────────────────────
  Stage 0  64           147×147 (2x down) 1,382,976      
  Stage 1  192           71×71  (4x down) 967,872        
  Stage 2  288           35×35  (9x down) 352,800        
  Stage 3  768           17×17  (18x down) 221,952        
  Stage 4  2048           8×8   (37x down) 131,072        
──────────────────────────────────────────────────────────────────────


Loading Model 3...


...


──────────────────────────────────────────────────────────────────────
  BACKBONE STRUCTURE ANALYSIS: ConvNeXtV2-Tiny
──────────────────────────────────────────────────────────────────────
  Input image size: 224×224
  Number of stages: 4

  Stage    Channels     Spatial size         Feature count  
  ──────────────────────────────────────────────────────────
  Stage 0  96            56×56  (4x down) 301,056        
  Stage 1  192           28×28  (8x down) 150,528        
  Stage 2  384           14×14  (16x down) 75,264         
  Stage 3  768            7×7   (32x down) 37,632         
──────────────────────────────────────────────────────────────────────


Loading Model 4...


model.safetensors:   0%|          | 0.00/81.1M [00:00<?, ?B/s]


──────────────────────────────────────────────────────────────────────
  BACKBONE STRUCTURE ANALYSIS: DenseNet-201
──────────────────────────────────────────────────────────────────────
  Input image size: 256×256
  Number of stages: 5

  Stage    Channels     Spatial size         Feature count  
  ──────────────────────────────────────────────────────────
  Stage 0  64           128×128 (2x down) 1,048,576      
  Stage 1  256           64×64  (4x down) 1,048,576      
  Stage 2  512           32×32  (8x down) 524,288        
  Stage 3  1792          16×16  (16x down) 458,752        
  Stage 4  1920           8×8   (32x down) 122,880        
──────────────────────────────────────────────────────────────────────


Loading Model 5...


model.safetensors:   0%|          | 0.00/100M [00:00<?, ?B/s]


──────────────────────────────────────────────────────────────────────
  BACKBONE STRUCTURE ANALYSIS: ResNeXt-50 (32×4d)
──────────────────────────────────────────────────────────────────────
  Input image size: 256×256
  Number of stages: 5

  Stage    Channels     Spatial size         Feature count  
  ──────────────────────────────────────────────────────────
  Stage 0  64           128×128 (2x down) 1,048,576      
  Stage 1  256           64×64  (4x down) 1,048,576      
  Stage 2  512           32×32  (8x down) 524,288        
  Stage 3  1024          16×16  (16x down) 262,144        
  Stage 4  2048           8×8   (32x down) 131,072        
──────────────────────────────────────────────────────────────────────


  QUICK REFERENCE: ALL BACKBONES

Model Name                # Stages     Channel Dimensions                                               
----------------------------------------------------------------------------------------------------
EfficientNet-B5           5  